# Conda Environment: 06_ALDEx2

---

# Library Imports

In [1]:
%%capture

# for python data manipulation
import pandas as pd

# for running R through Python
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri

# Load the rpy2 extension
%load_ext rpy2.ipython

# Load the library
importr('ALDEx2')

---

# Create Output Directories

In [2]:
!mkdir -p ../Datasets/ALDEx2

---

# Import Sample Metadata

In [3]:
sample_data = pd.read_csv("../Datasets/sample-data.csv")

---

# Format Data for ALDEx2

In [4]:
# Import stage 2 resident data (all weeks)
StageTwoResident = pd.read_csv("../Datasets/SequenceTables/StageTwoResident.csv", index_col= 0)

# Get sample names for stage 2 resident at start and end time points (weeks 2 and 8) per scale of fluctuation
coldata = sample_data[(sample_data['Stage'] == 2) & (sample_data['Active'] == 0) & (sample_data['Time point (week)'].isin([2,8]))]

lowT_metadata = coldata[coldata['Scale of fluctuation'] == 'lowT']
highT_metadata = coldata[coldata['Scale of fluctuation'] == 'highT']
TwoWeeks_metadata = coldata[coldata['Scale of fluctuation'] == '2Weeks']
OneWeek_metadata = coldata[coldata['Scale of fluctuation'] == '1Week']
TwoDays_metadata = coldata[coldata['Scale of fluctuation'] == '2Days']

# Subset sequence table by scale of fluctuation
lowT = StageTwoResident[lowT_metadata['Sample']]
highT = StageTwoResident[highT_metadata['Sample']]
TwoWeeks = StageTwoResident[TwoWeeks_metadata['Sample']]
OneWeek = StageTwoResident[OneWeek_metadata['Sample']]
TwoDays = StageTwoResident[TwoDays_metadata['Sample']]

# Export sequence tables for use in ALDEx2
lowT.to_csv("../Datasets/ALDEx2/lowT.csv", index= True)
highT.to_csv("../Datasets/ALDEx2/highT.csv", index= True)
TwoWeeks.to_csv("../Datasets/ALDEx2/TwoWeeks.csv", index= True)
OneWeek.to_csv("../Datasets/ALDEx2/OneWeek.csv", index= True)
TwoDays.to_csv("../Datasets/ALDEx2/TwoDays.csv", index= True)

---

# Calculate Difference in Per-Taxon Abundance Over Time Using ALDEx2

NOTE: The use of Monte Carlo simulations will results in quantitatively different but qualitatively similar numbers each time this is run. Refer to the supplementary data to get the exact fitness values that were used for each taxon in subsequent modelling.

In [5]:
%%capture

conditions= ['lowT', 'highT', 'TwoWeeks', 'OneWeek', 'TwoDays']

for condition in conditions:
    ro.r(f"""
    # Read in sequence table
    {condition} = read.csv('../Datasets/ALDEx2/{condition}.csv')
    rownames({condition}) = {condition}$ASV_ID
    {condition}$ASV_ID = NULL

    # create group vector
    group_vector  = c(rep("start",3), rep("end",3))

    # Run ALDEx2 estimation
    {condition}_result = aldex({condition}, group_vector, mc.samples=128, test="t", effect=TRUE)
    
    # Export results
    write.csv({condition}_result,"../Datasets/ALDEx2/{condition}_ALDEx2.csv")
    """)